# 03 — Evaluate Elora's Conditional CFM with Midpoint & RK4

Elora trained a **conditional** OT-CFM (class-conditioned, CFG-ready) on CIFAR-10 using Euler sampling.

Here we evaluate her best checkpoint (epoch 190) with **midpoint** and **rk4** ODE solvers from torchdiffeq.

**Key differences from our unconditional CFM:**
- `SimpleFlowUNet` with class conditioning (`forward(x, t, y)`)
- 10 classes + null label for classifier-free guidance
- Channel mult: 1x→2x→4x (vs our 1x→2x→2x→2x)
- `t_dim=256`, GroupNorm groups=8, `ema_decay=0.999`

**Run on Colab with GPU.**

In [ ]:
import math
import time
import json
import os
import subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from dataclasses import dataclass
import gc

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.inception import InceptionScore
except ImportError:
    subprocess.check_call(['pip', 'install', '-q', 'torchmetrics[image]'])
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.inception import InceptionScore

try:
    from torchdiffeq import odeint
except ImportError:
    subprocess.check_call(['pip', 'install', '-q', 'torchdiffeq'])
    from torchdiffeq import odeint

In [ ]:
DRIVE_DIR       = '/content/drive/MyDrive/flow-matching'
CFM_RUN_DIR     = f'{DRIVE_DIR}/runs/cfm_elora'
METRICS_DIR     = f'{CFM_RUN_DIR}/metrics'
EVAL_IMAGES_DIR = f'{CFM_RUN_DIR}/samples/eval'

os.makedirs(METRICS_DIR,     exist_ok=True)
os.makedirs(EVAL_IMAGES_DIR, exist_ok=True)

CHECKPOINT_PATH = f'{DRIVE_DIR}/runs/cfm/checkpoints/elora_epoch_190.pt'

NFE_POINTS      = [10, 20, 50, 100]
NUM_EVAL_IMAGES = 10000
EVAL_BATCH_SIZE = 256
EVAL_SEED       = 42
CFG_SCALE       = 1.0

---
## Step 1: Elora's Model Architecture

Her `SimpleFlowUNet` takes `(x, t, y)` — class-conditioned with classifier-free guidance support.

The model architecture must match exactly to load the checkpoint.

In [ ]:
def sinusoidal_time_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / max(half - 1, 1))
    angles = t[:, None] * freqs[None, :] * 2 * math.pi
    emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)
    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb


class SelfAttention2d(nn.Module):
    def __init__(self, ch: int):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.q = nn.Conv2d(ch, ch, kernel_size=1)
        self.k = nn.Conv2d(ch, ch, kernel_size=1)
        self.v = nn.Conv2d(ch, ch, kernel_size=1)
        self.proj = nn.Conv2d(ch, ch, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        h_ = self.norm(x)
        q = self.q(h_).reshape(b, c, h * w).permute(0, 2, 1)
        k = self.k(h_).reshape(b, c, h * w)
        attn = torch.bmm(q, k) * (c ** -0.5)
        attn = torch.softmax(attn, dim=-1)
        v = self.v(h_).reshape(b, c, h * w).permute(0, 2, 1)
        out = torch.bmm(attn, v).permute(0, 2, 1).reshape(b, c, h, w)
        return x + self.proj(out)


class TimeResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, t_dim: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.y_proj = nn.Linear(t_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb, y_emb):
        h = self.conv1(x)
        h = self.norm1(h)
        h = h + self.t_proj(t_emb)[:, :, None, None] + self.y_proj(y_emb)[:, :, None, None]
        h = F.silu(h)
        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)
        return h + self.skip(x)


class SimpleFlowUNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=128, t_dim=256, num_classes=10,
                 use_attn_16=True, use_attn_8=True):
        super().__init__()
        self.t_dim = t_dim
        self.num_classes = num_classes
        self.null_label = num_classes

        self.time_mlp = nn.Sequential(nn.Linear(t_dim, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))
        self.class_emb = nn.Embedding(num_classes + 1, t_dim)

        self.enc1a = TimeResBlock(in_ch, base_ch, t_dim)
        self.enc1b = TimeResBlock(base_ch, base_ch, t_dim)

        self.down1 = nn.Conv2d(base_ch, base_ch * 2, 4, stride=2, padding=1)
        self.enc2a = TimeResBlock(base_ch * 2, base_ch * 2, t_dim)
        self.enc2_attn1 = SelfAttention2d(base_ch * 2) if use_attn_16 else nn.Identity()
        self.enc2b = TimeResBlock(base_ch * 2, base_ch * 2, t_dim)
        self.enc2_attn2 = SelfAttention2d(base_ch * 2) if use_attn_16 else nn.Identity()

        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 4, 4, stride=2, padding=1)
        self.enc3a = TimeResBlock(base_ch * 4, base_ch * 4, t_dim)
        self.enc3_attn = SelfAttention2d(base_ch * 4) if use_attn_8 else nn.Identity()
        self.enc3b = TimeResBlock(base_ch * 4, base_ch * 4, t_dim)

        self.mid1 = TimeResBlock(base_ch * 4, base_ch * 4, t_dim)
        self.mid_attn = SelfAttention2d(base_ch * 4) if use_attn_8 else nn.Identity()
        self.mid2 = TimeResBlock(base_ch * 4, base_ch * 4, t_dim)

        self.dec3a = TimeResBlock(base_ch * 8, base_ch * 4, t_dim)
        self.dec3b = TimeResBlock(base_ch * 4, base_ch * 4, t_dim)
        self.dec3_attn = SelfAttention2d(base_ch * 4) if use_attn_8 else nn.Identity()

        self.dec2a = TimeResBlock(base_ch * 6, base_ch * 2, t_dim)
        self.dec2_attn1 = SelfAttention2d(base_ch * 2) if use_attn_16 else nn.Identity()
        self.dec2b = TimeResBlock(base_ch * 2, base_ch * 2, t_dim)
        self.dec2_attn2 = SelfAttention2d(base_ch * 2) if use_attn_16 else nn.Identity()

        self.dec1a = TimeResBlock(base_ch * 3, base_ch, t_dim)
        self.dec1b = TimeResBlock(base_ch, base_ch, t_dim)
        self.out = nn.Conv2d(base_ch, in_ch, 3, padding=1)

    def forward(self, x, t, y=None):
        if y is None:
            y = torch.full((x.size(0),), self.null_label, device=x.device, dtype=torch.long)

        t_emb = self.time_mlp(sinusoidal_time_embedding(t, self.t_dim))
        y_emb = self.class_emb(y.long())

        s1 = self.enc1a(x, t_emb, y_emb)
        s1 = self.enc1b(s1, t_emb, y_emb)

        h = self.down1(s1)
        s2 = self.enc2a(h, t_emb, y_emb)
        s2 = self.enc2_attn1(s2)
        s2 = self.enc2b(s2, t_emb, y_emb)
        s2 = self.enc2_attn2(s2)

        h = self.down2(s2)
        s3 = self.enc3a(h, t_emb, y_emb)
        s3 = self.enc3_attn(s3)
        s3 = self.enc3b(s3, t_emb, y_emb)

        h = self.mid1(s3, t_emb, y_emb)
        h = self.mid_attn(h)
        h = self.mid2(h, t_emb, y_emb)

        h = torch.cat([h, s3], dim=1)
        h = self.dec3a(h, t_emb, y_emb)
        h = self.dec3b(h, t_emb, y_emb)
        h = self.dec3_attn(h)

        h = F.interpolate(h, scale_factor=2, mode='nearest')
        h = torch.cat([h, s2], dim=1)
        h = self.dec2a(h, t_emb, y_emb)
        h = self.dec2_attn1(h)
        h = self.dec2b(h, t_emb, y_emb)
        h = self.dec2_attn2(h)

        h = F.interpolate(h, scale_factor=2, mode='nearest')
        h = torch.cat([h, s1], dim=1)
        h = self.dec1a(h, t_emb, y_emb)
        h = self.dec1b(h, t_emb, y_emb)
        return self.out(h)

---
## Step 2: ODE Sampler (torchdiffeq)

Wrap the conditional model in a closure that captures class labels, then pass to `odeint`.

For CFG: compute both conditional and unconditional velocities, then combine.

In [ ]:
@torch.no_grad()
def cfm_sample_conditional(model, n_samples, device, method='midpoint', nfe=50,
                           labels=None, cfg_scale=1.0, seed=42):
    """Sample from a conditional CFM using torchdiffeq ODE solvers."""
    model.eval()
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    x0 = torch.randn(n_samples, 3, 32, 32, device=device, generator=gen)

    if labels is None:
        labels = torch.randint(0, model.num_classes, (n_samples,), device=device, generator=gen)
    else:
        labels = labels.to(device).long()
        if labels.shape[0] != n_samples:
            labels = labels.repeat((n_samples + labels.shape[0] - 1) // labels.shape[0])[:n_samples]

    null_labels = torch.full_like(labels, model.null_label)

    def ode_fn(t, x):
        t_batch = torch.full((n_samples,), t.item(), device=device)
        v_cond = model(x, t_batch, labels)
        if cfg_scale != 1.0:
            v_uncond = model(x, t_batch, null_labels)
            return v_uncond + cfg_scale * (v_cond - v_uncond)
        return v_cond

    t_span = torch.tensor([0.0, 1.0], device=device)
    nfe_per_step = {'midpoint': 2, 'rk4': 4}[method]
    step_size = nfe_per_step / nfe

    x = odeint(ode_fn, x0, t_span, method=method,
               options={'step_size': step_size})[-1]
    return x.clamp(-1, 1)

---
## Step 3: Load Elora's checkpoint

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
print(f'Checkpoint: epoch {ckpt["epoch"]}, loss={ckpt["epoch_loss"]:.4f}')
print(f'Config: {ckpt["config"]}')

model_cfg = ckpt['config']
model = SimpleFlowUNet(
    in_ch=3,
    base_ch=model_cfg['base_ch'],
    t_dim=model_cfg['t_dim'],
    num_classes=model_cfg['num_classes'],
    use_attn_16=model_cfg['use_attn_16'],
    use_attn_8=model_cfg['use_attn_8'],
).to(device)

# Load EMA weights (stored as full model state_dict)
model.load_state_dict(ckpt['ema_state_dict'])
model.eval()
print(f'Loaded EMA weights — {sum(p.numel() for p in model.parameters()):,} params')

---
## Step 4: Generate images + compute metrics

In [ ]:
def get_real_images(n, batch_size=256):
    dataset = datasets.CIFAR10(root='./data', train=True, download=True,
                               transform=transforms.ToTensor())
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    imgs = []
    for x, _ in tqdm(loader, desc='Loading real images'):
        imgs.append((x * 255).byte())
        if sum(b.shape[0] for b in imgs) >= n:
            break
    return torch.cat(imgs)[:n]


def generate_images(model, n, batch_size, method='midpoint', nfe=50,
                    cfg_scale=1.0, seed=42):
    all_images = []
    total_time = 0.0
    generated  = 0
    batch_idx  = 0

    while generated < n:
        bs = min(batch_size, n - generated)
        t_start = time.time()
        imgs = cfm_sample_conditional(
            model, bs, device, method=method, nfe=nfe,
            cfg_scale=cfg_scale, seed=seed + batch_idx
        )
        total_time += time.time() - t_start
        imgs = ((imgs.cpu() + 1) / 2).clamp(0, 1)
        imgs = (imgs * 255).byte()
        all_images.append(imgs)
        generated += bs
        batch_idx += 1
        torch.cuda.empty_cache()

    return torch.cat(all_images)[:n], total_time


def compute_fid_is(real_imgs, fake_imgs):
    bs = 256
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)
    for i in range(0, len(real_imgs), bs):
        fid_metric.update(real_imgs[i:i+bs].to(device), real=True)
    for i in range(0, len(fake_imgs), bs):
        fid_metric.update(fake_imgs[i:i+bs].to(device), real=False)
    fid_score = fid_metric.compute().item()
    del fid_metric

    is_metric = InceptionScore().to(device)
    for i in range(0, len(fake_imgs), bs):
        is_metric.update(fake_imgs[i:i+bs].to(device))
    is_mean, is_std = is_metric.compute()
    is_mean, is_std = is_mean.item(), is_std.item()
    del is_metric

    torch.cuda.empty_cache()
    return fid_score, is_mean, is_std

In [ ]:
print('Loading real CIFAR-10 images...')
real_imgs = get_real_images(NUM_EVAL_IMAGES)
print(f'Real images: {real_imgs.shape}, dtype={real_imgs.dtype}')

all_results = []

for method in ['midpoint', 'rk4']:
    for nfe in NFE_POINTS:
        if method == 'rk4' and nfe % 4 != 0:
            print(f'\nSkipping {method} NFE={nfe} (not divisible by 4)')
            continue

        print(f'\n{"="*50}')
        print(f'Evaluating {method} at NFE={nfe}')
        print(f'{"="*50}')

        fake_imgs, total_sec = generate_images(
            model, n=NUM_EVAL_IMAGES, batch_size=EVAL_BATCH_SIZE,
            method=method, nfe=nfe, cfg_scale=CFG_SCALE, seed=EVAL_SEED
        )

        nfe_dir = f'{EVAL_IMAGES_DIR}/{method}_nfe_{nfe}'
        os.makedirs(nfe_dir, exist_ok=True)
        grid = torchvision.utils.make_grid(fake_imgs[:64].float() / 255, nrow=8, padding=2)
        torchvision.utils.save_image(grid, f'{nfe_dir}/sample_grid.png')

        fid, is_mean, is_std = compute_fid_is(real_imgs, fake_imgs)
        sec_per_image = total_sec / NUM_EVAL_IMAGES

        result = {
            'model': 'elora-conditional-cfm',
            'checkpoint': 'elora_epoch_190.pt',
            'nfe': nfe,
            'solver_sampler': method,
            'cfg_scale': CFG_SCALE,
            'num_generated': NUM_EVAL_IMAGES,
            'fid': round(fid, 4),
            'is_mean': round(is_mean, 4),
            'is_std': round(is_std, 4),
            'sec_per_image': round(sec_per_image, 6),
            'total_sampling_sec': round(total_sec, 2),
            'seed': EVAL_SEED,
        }
        all_results.append(result)
        print(f'  FID={fid:.2f}  IS={is_mean:.2f}\u00b1{is_std:.2f}  {sec_per_image*1000:.1f}ms/img')

        del fake_imgs
        torch.cuda.empty_cache()
        gc.collect()

print('\nAll evaluations done!')

---
## Step 5: Save metrics

In [ ]:
import datetime

jsonl_path = f'{METRICS_DIR}/metrics_summary.jsonl'
with open(jsonl_path, 'a') as f:
    for r in all_results:
        r['evaluated_at'] = datetime.datetime.now().isoformat()
        f.write(json.dumps(r) + '\n')

json_path = f'{METRICS_DIR}/metrics_summary.json'
with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)

fid_results = [{'nfe': r['nfe'], 'solver_sampler': r['solver_sampler'],
                'fid': r['fid'], 'is_mean': r['is_mean'], 'is_std': r['is_std']} for r in all_results]
with open(f'{METRICS_DIR}/fid_results.json', 'w') as f:
    json.dump(fid_results, f, indent=2)

runtime_results = [{'nfe': r['nfe'], 'solver_sampler': r['solver_sampler'],
                    'sec_per_image': r['sec_per_image'],
                    'total_sampling_sec': r['total_sampling_sec']} for r in all_results]
with open(f'{METRICS_DIR}/runtime_results.json', 'w') as f:
    json.dump(runtime_results, f, indent=2)

print(f'Metrics saved to {METRICS_DIR}')

---
## Step 6: Visualize results

Compare midpoint and rk4 solvers on Elora's model, plus her original Euler results.

In [ ]:
# Elora's Euler results from Experiments.md (epoch 190, cfg_scale=1.0)
euler_results = [
    {'nfe': 10,  'solver_sampler': 'euler', 'fid': 21.0696, 'is_mean': 7.814, 'sec_per_image': 0.003371},
    {'nfe': 20,  'solver_sampler': 'euler', 'fid': 16.6913, 'is_mean': 7.954, 'sec_per_image': 0.005854},
    {'nfe': 50,  'solver_sampler': 'euler', 'fid': 13.7815, 'is_mean': 8.076, 'sec_per_image': 0.013311},
    {'nfe': 100, 'solver_sampler': 'euler', 'fid': 13.1867, 'is_mean': 8.272, 'sec_per_image': 0.025744},
]

midpoint_results = [r for r in all_results if r['solver_sampler'] == 'midpoint']
rk4_results      = [r for r in all_results if r['solver_sampler'] == 'rk4']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Elora's Conditional CFM: Euler vs Midpoint vs RK4", fontsize=13)

# FID vs NFE
axes[0].plot([r['nfe'] for r in euler_results], [r['fid'] for r in euler_results],
             '^-', color='forestgreen', linewidth=2, markersize=8, label='Euler (original)')
axes[0].plot([r['nfe'] for r in midpoint_results], [r['fid'] for r in midpoint_results],
             'o-', color='steelblue', linewidth=2, markersize=8, label='Midpoint')
axes[0].plot([r['nfe'] for r in rk4_results], [r['fid'] for r in rk4_results],
             's-', color='tomato', linewidth=2, markersize=8, label='RK4')
axes[0].set_xlabel('NFE')
axes[0].set_ylabel('FID \u2193')
axes[0].set_title('FID vs NFE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Runtime vs NFE
axes[1].plot([r['nfe'] for r in euler_results],
             [r['sec_per_image']*1000 for r in euler_results],
             '^-', color='forestgreen', linewidth=2, markersize=8, label='Euler (original)')
axes[1].plot([r['nfe'] for r in midpoint_results],
             [r['sec_per_image']*1000 for r in midpoint_results],
             'o-', color='steelblue', linewidth=2, markersize=8, label='Midpoint')
axes[1].plot([r['nfe'] for r in rk4_results],
             [r['sec_per_image']*1000 for r in rk4_results],
             's-', color='tomato', linewidth=2, markersize=8, label='RK4')
axes[1].set_xlabel('NFE')
axes[1].set_ylabel('ms / image \u2193')
axes[1].set_title('Runtime vs NFE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{METRICS_DIR}/fid_vs_nfe.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
combined = all_results + euler_results

print(f'\n{"Solver":<12} {"NFE":<6} {"FID":<8} {"IS":<12} {"ms/img":<10}')
print('-' * 52)
for r in sorted(combined, key=lambda x: (x['solver_sampler'], x['nfe'])):
    fid_str = f'{r["fid"]:.2f}'
    is_str  = f'{r.get("is_mean", 0):.2f}' + (f'\u00b1{r["is_std"]:.2f}' if 'is_std' in r else '')
    ms_str  = f'{r["sec_per_image"]*1000:.1f}'
    print(f'{r["solver_sampler"]:<12} {r["nfe"]:<6} {fid_str:<8} {is_str:<12} {ms_str:<10}')